# WolBanking77 — test du modèle GitHub

Ce notebook ne réalise **aucun entraînement**. Il clone le projet GitHub, charge `models/baseline_5k_split.joblib` dans la variable `model`, puis effectue des tests.

## 1. Cloner le projet et charger le modèle

In [ ]:
from pathlib import Path
import joblib

REPO_URL = 'https://github.com/alimar440/Projet-WolBanking77.git'
PROJECT_DIR = Path('/content/WolBanking77')
MODEL_PATH = PROJECT_DIR / 'models' / 'baseline_5k_split.joblib'

if not PROJECT_DIR.exists():
    !git clone $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull

assert MODEL_PATH.exists(), f'Modèle introuvable : {MODEL_PATH}'
model = joblib.load(MODEL_PATH)

print('Projet cloné dans :', PROJECT_DIR)
print('Modèle chargé :', MODEL_PATH)
print('Variable model :', type(model).__name__)

## 2. Fonction de prédiction

Le modèle retourne l'intention prédite. Les scores affichés sont les scores de décision du SVM : ce ne sont pas des probabilités.

In [ ]:
import numpy as np
import pandas as pd

def predict_intent(text, top_k=3):
    scores = model.decision_function([text])[0]
    best_indices = np.argsort(scores)[-top_k:][::-1]
    return [
        {'intention': model.classes_[index], 'score_svm': round(float(scores[index]), 4)}
        for index in best_indices
    ]

def show_prediction(text):
    print(f'Phrase : {text}')
    for result in predict_intent(text):
        print(f"  - {result['intention']} (score : {result['score_svm']})")

## 3. Tester vos propres phrases

Modifiez les phrases ci-dessous puis exécutez la cellule.

In [ ]:
phrases_a_tester = [
    'Xamuma lutax ñu bañ sama payoor?',   
    'Bëgg naa xam sama solde ci bank bi.',
    'Sama karte bancaire dafa réer.',
]

for phrase in phrases_a_tester:
    show_prediction(phrase)
    print()

## 4. Tester des exemples du CSV de test

Cette cellule compare la prédiction à la vraie intention pour cinq exemples aléatoires.

In [ ]:
TEST_PATH = PROJECT_DIR / 'data' / '5k_split' / 'test' / 'test.csv'
test_df = pd.read_csv(TEST_PATH)
examples = test_df.sample(5, random_state=42)

for _, row in examples.iterrows():
    prediction = predict_intent(row['input_wo'], top_k=1)[0]
    print(f"Texte : {row['input_wo']}")
    print(f"Vraie intention : {row['label']}")
    print(f"Prédiction : {prediction['intention']} (score : {prediction['score_svm']})")
    print('-' * 80)

## 5. Mesurer la performance sur tout le jeu de test

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

predictions = model.predict(test_df['input_wo'])
accuracy = accuracy_score(test_df['label'], predictions)
macro_f1 = f1_score(test_df['label'], predictions, average='macro', zero_division=0)

print(f'Accuracy : {accuracy:.2%}')
print(f'Macro-F1 : {macro_f1:.2%}')